# 2. Load silver tables

In [0]:
delta_base = "hr_catalogue_business_case.hr_raw"

df_absences  = spark.table(delta_base + ".absences")
df_contracts = spark.table(delta_base + ".contracts")
df_salary    = spark.table(delta_base + ".salary")
df_workplan  = spark.table(delta_base + ".workplan")
df_postcodes = spark.table(delta_base + ".postcodes")
df_abs_types = spark.table(delta_base + ".absence_types")

print("Silver tables loaded")

In [0]:
df_contracts.printSchema()

## 2.1 : Creation of unique key to merge datasets

In [0]:
dfs = {
    "absences": df_absences,
    "contracts": df_contracts,
    "salary": df_salary,
    "workplan": df_workplan,
    "postcodes": df_postcodes,
    "abs_types": df_abs_types
}

for name, df in dfs.items():
    has_fdcp = "FDCP" in df.columns
    print(name + " — FDCP: " + str(has_fdcp))

### 2.1.1. Contracts

In [0]:
from pyspark.sql.functions import col, concat_ws

df_contracts_FDCP = df_contracts.withColumn(
    "FDCP",
    concat_ws("|",
        col("Firm_ID").cast("string"),
        col("Department_ID").cast("string"),
        col("Category_ID").cast("string"),
        col("Person_ID").cast("string")
    )
)

df_contracts_FDCP.select("FDCP", "Firm_ID", "Department_ID", "Category_ID", "Person_ID").show(5)

### 2.1.2. Absences

In [0]:
df_absences_FDCP = df_absences.withColumn(
    "FDCP",
    concat_ws("|",
        col("Firm_ID").cast("string"),
        col("Department_ID").cast("string"),
        col("Category_ID").cast("string"),
        col("Person_ID").cast("string")
    )
)

df_absences_FDCP.select("FDCP", "Firm_ID", "Department_ID", "Category_ID", "Person_ID").show(5)

## 2.2. Feature engineering : add features for model

In [0]:
from pyspark.sql.functions import current_date, datediff, floor

df_contracts_enriched = df_contracts_FDCP \
    .withColumn("Age",
        floor(datediff(current_date(), col("Birth_Date")) / 365.25)
    ) \
    .withColumn("Seniority_Years",
        floor(datediff(current_date(), col("Company_Start_Date")) / 365.25)
    )

df_contracts_enriched.select("FDCP", "Age", "Seniority_Years").show(5)

### 2.3 Star schema - Identifying dim and fact tables

In [0]:
## Identify first range of dates
from pyspark.sql.functions import min, max

print("=== absences ===")
df_absences.select(min("Date"), max("Date")).show()

print("=== contracts ===")
df_contracts.select(min("Contract_Start_Date"), max("Contract_Start_Date")).show()

print("=== salary ===")
df_salary.select(min("Period"), max("Period")).show()

In [0]:
dim_employee = df_contracts_enriched.select(
    "FDCP",
    "Gender",
    "Nationality",
    "Contract_Type",
    "Age",
    "Seniority_Years",
    "Contract_Start_Date",
    "Contract_End_Date",
    "Company_Start_Date",
    "Contract_Terminatio_Reason",
    "Contract_ZIP_Code"
)

dim_employee.show(3)

In [0]:
dim_region = df_postcodes.select(
    "PostCode",
    "Region_Code",
    "Region"
)

dim_region.show(3)

In [0]:
dim_abs_type = df_abs_types.select(
    "Type_Absence",
    "Type_Absence_FR"
)

dim_abs_type.show()

In [0]:
dim_date = spark.range(1).select(
    explode(
        sequence(to_date(lit("2016-01-01")), to_date(lit("2018-12-31")), expr("interval 1 day"))
    ).alias("Date")
).select(
    "Date",
    year("Date").alias("Year"),
    quarter("Date").alias("Quarter"),
    month("Date").alias("Month"),
    dayofmonth("Date").alias("Day"),
    date_format("Date", "MMMM").alias("Month_Name")
)

dim_date.show(3)

In [0]:
qty_cols = [c for c in df_absences_FDCP.columns if c.startswith("Qty_")]
freq_cols = [c for c in df_absences_FDCP.columns if c.startswith("Freq_")]

fact_absences = df_absences_FDCP.join(
    dim_employee.select("FDCP", "Contract_ZIP_Code"),
    on="FDCP",
    how="left"
).select(
    ["FDCP", "Date", "Year", "Quarter", "Month"] + qty_cols + freq_cols + ["Contract_ZIP_Code"]
)

print("Columns: " + str(len(fact_absences.columns)))
fact_absences.show(3)

In [0]:
%sql
-- CREATE SCHEMA IF NOT EXISTS hr_catalogue_business_case.hr_gold

In [0]:
gold_base = "hr_catalogue_business_case.hr_gold"

dim_employee.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".dim_employee")
dim_region.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".dim_region")
dim_abs_type.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".dim_abs_type")
dim_date.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".dim_date")
fact_absences.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".fact_absences")
df_salary.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".salary")
df_workplan.write.format("delta").mode("overwrite").saveAsTable(gold_base + ".workplan")

print("All Gold tables saved successfully")

In [0]:
gold_base = "hr_catalogue_business_case.hr_gold"

tables = ["dim_employee", "dim_region", "dim_abs_type", "dim_date", "fact_absences", "salary", "workplan"]

for t in tables:
    count = spark.table(gold_base + "." + t).count()
    print(t + " — rows: " + str(count))

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

for t in tables:
    df = spark.table(gold_base + "." + t)
    print("\n=== " + t + " ===")
    null_counts = df.select([
        spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    for col_name, null_count in null_counts.items():
        if null_count > 0:
            print("  " + col_name + " : " + str(null_count) + " nulls")